In [ ]:
### Script for plotting the SED_PPE hazard curves
import h5py as h5
import numpy as np
import os
import matplotlib.pyplot as plt
import geopandas as gpd

In [ ]:
results_dir = "../results/CFM/sites_c_MDEz/N34_b0959_C42_S10_TI_geologic_c_MDEz_sites"
plots_dir = os.path.join(os.path.dirname(results_dir), "SED_PPE_plots")
os.makedirs(plots_dir, exist_ok=True)

site_geojson = "../crustal/discretised_CFM/crustal_site_locations_10km_coast.geojson"

years = 7500

sites = gpd.read_file(site_geojson)
sites = sites.iloc[:1]
sites

In [ ]:
with h5.File(f"{results_dir}/1671156_5325618.h5", 'r') as site_h5:
    print(site_h5.keys())


In [ ]:
for siteId in sites.siteId:
    try:
        with h5.File(f"{results_dir}/{siteId}.h5", 'r') as site_h5:
            up_SED = site_h5[str(years)]['SED_PPE']['up_SED_PPE'][:]
            down_SED = site_h5[str(years)]['SED_PPE']['down_SED_PPE'][:]
            up_SED_thresh = site_h5[str(years)]['SED_PPE']['up_SED_thresh'][:]
            down_SED_thresh = -site_h5[str(years)]['SED_PPE']['down_SED_thresh'][:]
            PPE_start, PPE_step = site_h5[str(years)]['thresh_para'][0], site_h5[str(years)]['thresh_para'][-1]
            up_PPE = site_h5[str(years)]['exceedance_probs_up'][:]
            down_PPE = site_h5[str(years)]['exceedance_probs_down'][:]
            up_PPE_thresh = np.arange(PPE_start, PPE_start + PPE_step * up_PPE.shape[0], PPE_step)
            down_PPE_thresh = np.arange(PPE_start, PPE_start + PPE_step * down_PPE.shape[0], PPE_step)
            plt.plot(up_SED_thresh, up_SED * 100, 'r', label="Up SED")
            plt.plot(down_SED_thresh, down_SED * 100, 'b', label="Down SED")
            plt.plot(up_PPE_thresh, up_PPE * 100, 'r:', label="Up Cumulative PPE")
            plt.plot(down_PPE_thresh, down_PPE * 100, 'b:', label="Down Cumulative PPE")
            plt.xlabel("PPE threshold (m)")
            plt.ylabel("Probability of Exceedance (%)")
            plt.xscale('log')
            plt.yscale('log')
            plt.ylim([0.1, 110])
            plt.xlim([0.1, 30])
            plt.title(f'Site: {siteId}, {years} yrs with 100,000 samples')
            plt.legend()
            plt.savefig(os.path.join(plots_dir, f"{siteId}_SED.png"))
            plt.close()
    except:
        print(f"[ERROR]: {siteId}")
        try:
            plt.close()
        except:
            pass


